# Composite Entity / Identity Features

This experiment tests four small, interpretable composite identities with the existing fixed LightGBM baseline. Selection uses three expanding chronological folds inside TRAIN. External VALIDATION is evaluated exactly once after selection; HOLDOUT is never assigned or accessed.

In [1]:
import gc
import json
import sys
import time
from pathlib import Path

import lightgbm as lgb
import mlflow
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.preprocessing import StandardScaler

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from fraud_detection.data import (
    load_train_identity,
    load_train_transaction,
    merge_train_data,
    split_data_chronologically,
    validate_training_data,
)
from fraud_detection.features import (
    DEFAULT_COMPOSITE_IDENTITIES,
    BehavioralFeatureEngineer,
    CompositeFeatureEngineer,
    Phase1FeatureEngineer,
)
from fraud_detection.models import (
    TrainingOnlyPreprocessor,
    evaluate_probabilities,
    make_expanding_time_folds,
)

notebook_started = time.perf_counter()

## Composite definitions and leakage safeguards

Each component is encoded as an explicit missing token or a length-prefixed nonmissing value before joining. Composite frequency maps are learned from each fold's fit rows only. Historical statistics use complete timestamp batches, so transactions at time `T` see only `TransactionDT < T`; labels are never inputs.

In [2]:
raw_dir = PROJECT_ROOT / "data" / "raw"
transaction = load_train_transaction(raw_dir)
identity = load_train_identity(raw_dir)
merged = merge_train_data(transaction, identity)
split = split_data_chronologically(merged)
train = split.train
validate_training_data(train)
del transaction, identity, merged
gc.collect()

folds = make_expanding_time_folds(train, n_splits=3, initial_train_fraction=0.70)
fold_design = pd.DataFrame(
    [
        {
            "Fold": number,
            "Fit Rows": len(fold.train_indices),
            "Evaluation Rows": len(fold.evaluation_indices),
            "Fit DT Max": train.iloc[fold.train_indices]["TransactionDT"].max(),
            "Evaluation DT Min": train.iloc[fold.evaluation_indices]["TransactionDT"].min(),
            "Evaluation DT Max": train.iloc[fold.evaluation_indices]["TransactionDT"].max(),
        }
        for number, fold in enumerate(folds, start=1)
    ]
)

lightgbm_parameters = {
    "learning_rate": 0.1,
    "n_estimators": 400,
    "num_leaves": 63,
    "class_weight": "balanced",
    "random_state": 42,
    "n_jobs": -1,
    "verbosity": -1,
}
feature_sets = (
    "baseline",
    "baseline_plus_composite_frequency",
    "baseline_plus_composite_history",
    "baseline_plus_all_composite",
)
identity_definitions = {
    identity.name: identity.columns for identity in DEFAULT_COMPOSITE_IDENTITIES
}
display(fold_design)
display(pd.Series(identity_definitions, name="components"))
print("External validation status: not assigned or accessed before CV selection")
print("Final holdout status: not assigned or accessed")

,Fold,Fit Rows,Evaluation Rows,Fit DT Max,Evaluation DT Min,Evaluation DT Max
0,1,289364,41338,7135483,7135609,8121450
1,2,330702,41338,8121450,8121466,9247568
2,3,372040,41338,9247568,9247583,10437996


card1_card2_addr1                         (card1, card2, addr1)
card1_addr1_p_emaildomain         (card1, addr1, P_emaildomain)
card1_card2_addr1_productcd    (card1, card2, addr1, ProductCD)
card1_addr1_deviceinfo               (card1, addr1, DeviceInfo)
Name: components, dtype: object

External validation status: not assigned or accessed before CV selection
Final holdout status: not assigned or accessed


## Fold-local feature preparation

The existing raw-plus-frequency and five historical-amount baseline is rebuilt within every fold. The composite transformer is also fitted independently per fold. One combined preprocessed matrix is created per fold, then column subsets isolate each ablation.

In [3]:
def prepare_feature_pair(fit_frame, evaluation_frame):
    started = time.perf_counter()
    predictor_columns = [
        column for column in fit_frame.columns if column not in ("isFraud", "TransactionID")
    ]
    fit_predictors = fit_frame.loc[:, predictor_columns]
    evaluation_predictors = evaluation_frame.loc[:, predictor_columns]
    frequency_engineer = Phase1FeatureEngineer(
        include_amount_time=False,
        include_missing_indicators=False,
        include_frequency_encodings=True,
    ).fit(fit_predictors)
    frequency_fit = frequency_engineer.transform(fit_predictors)
    frequency_evaluation = frequency_engineer.transform(evaluation_predictors)

    behavior_columns = [
        column
        for column in (
            "TransactionID",
            "TransactionDT",
            "TransactionAmt",
            "card1",
            "addr1",
            "DeviceInfo",
            "P_emaildomain",
        )
        if column in fit_frame.columns
    ]
    amount_engineer = BehavioralFeatureEngineer(
        include_counts=False,
        include_amounts=True,
        include_recency=False,
        include_velocity=False,
    )
    amount_fit = amount_engineer.fit_transform_features(fit_frame.loc[:, behavior_columns])
    amount_evaluation = amount_engineer.transform_features(
        evaluation_frame.loc[:, behavior_columns], update_state=True
    )
    baseline_fit = pd.concat([frequency_fit, amount_fit], axis=1)
    baseline_evaluation = pd.concat([frequency_evaluation, amount_evaluation], axis=1)

    composite_engineer = CompositeFeatureEngineer()
    composite_fit = composite_engineer.fit_transform_features(fit_frame)
    composite_evaluation = composite_engineer.transform_features(
        evaluation_frame, update_state=True
    )
    combined_fit = pd.concat([baseline_fit, composite_fit], axis=1)
    combined_evaluation = pd.concat([baseline_evaluation, composite_evaluation], axis=1)
    preprocessor = TrainingOnlyPreprocessor()
    x_fit = preprocessor.fit_transform(combined_fit)
    x_evaluation = preprocessor.transform(combined_evaluation)
    scaler = StandardScaler(copy=False)
    x_fit = scaler.fit_transform(x_fit)
    x_evaluation = scaler.transform(x_evaluation)

    baseline_names = tuple(baseline_fit.columns)
    frequency_names = composite_engineer.feature_groups_["composite_frequency"]
    history_names = composite_engineer.feature_groups_["composite_history"]
    names_by_set = {
        "baseline": baseline_names,
        "baseline_plus_composite_frequency": baseline_names + frequency_names,
        "baseline_plus_composite_history": baseline_names + history_names,
        "baseline_plus_all_composite": baseline_names + frequency_names + history_names,
    }
    positions = {name: index for index, name in enumerate(preprocessor.feature_names_)}
    indices_by_set = {
        name: np.asarray([positions[column] for column in columns], dtype=np.int32)
        for name, columns in names_by_set.items()
    }
    memory_mib = (
        composite_fit.memory_usage(deep=True).sum()
        + composite_evaluation.memory_usage(deep=True).sum()
        + x_fit.nbytes
        + x_evaluation.nbytes
    ) / 1024**2
    seconds = time.perf_counter() - started
    return x_fit, x_evaluation, indices_by_set, names_by_set, seconds, memory_mib

## TRAIN-only temporal CV and ablation

In [4]:
tracking_uri = f"sqlite:///{(PROJECT_ROOT / 'model_comparison.db').resolve().as_posix()}"
mlflow.set_tracking_uri(tracking_uri)
mlflow.set_experiment("fraud-detection-composite-features")
cv_metrics = {name: [] for name in feature_sets}
feature_counts = {}
preparation_seconds = []
prepared_memory_mib = []

for fold_number, fold in enumerate(folds, start=1):
    fit_frame = train.iloc[fold.train_indices]
    evaluation_frame = train.iloc[fold.evaluation_indices]
    (
        x_fit,
        x_evaluation,
        indices_by_set,
        names_by_set,
        feature_seconds,
        memory_mib,
    ) = prepare_feature_pair(fit_frame, evaluation_frame)
    preparation_seconds.append(feature_seconds)
    prepared_memory_mib.append(memory_mib)
    y_fit = fit_frame["isFraud"].astype("int8")
    y_evaluation = evaluation_frame["isFraud"].astype("int8")

    for feature_set in feature_sets:
        selected = indices_by_set[feature_set]
        model = lgb.LGBMClassifier(**lightgbm_parameters)
        model.fit(x_fit[:, selected], y_fit)
        probabilities = model.predict_proba(x_evaluation[:, selected])[:, 1]
        metrics = evaluate_probabilities(y_evaluation, probabilities)
        cv_metrics[feature_set].append(metrics)
        feature_counts[feature_set] = len(names_by_set[feature_set])
        print(f"fold={fold_number} feature_set={feature_set} PR-AUC={metrics.pr_auc:.6f}")
        del model, probabilities
    del x_fit, x_evaluation, indices_by_set, names_by_set
    gc.collect()

cv_rows = []
cv_run_ids = {}
for feature_set in feature_sets:
    fold_pr_auc = [metric.pr_auc for metric in cv_metrics[feature_set]]
    fold_roc_auc = [metric.roc_auc for metric in cv_metrics[feature_set]]
    mean_pr_auc = float(np.mean(fold_pr_auc))
    std_pr_auc = float(np.std(fold_pr_auc, ddof=0))
    mean_roc_auc = float(np.mean(fold_roc_auc))
    with mlflow.start_run(run_name=f"composite-cv-{feature_set}") as run:
        mlflow.set_tags(
            {
                "stage": "composite-feature-engineering",
                "dataset": "ieee-cis",
                "selection": "temporal-cv",
            }
        )
        mlflow.log_params(
            {
                "composite_feature_set": feature_set,
                "composite_identities": json.dumps(identity_definitions),
                "feature_count": feature_counts[feature_set],
                **lightgbm_parameters,
            }
        )
        mlflow.log_metrics(
            {
                "fold_1_pr_auc": fold_pr_auc[0],
                "fold_2_pr_auc": fold_pr_auc[1],
                "fold_3_pr_auc": fold_pr_auc[2],
                "mean_cv_pr_auc": mean_pr_auc,
                "std_cv_pr_auc": std_pr_auc,
                "mean_cv_roc_auc": mean_roc_auc,
            }
        )
        cv_run_ids[feature_set] = run.info.run_id
    cv_rows.append(
        {
            "Feature Set": feature_set,
            "Feature Count": feature_counts[feature_set],
            "Fold PR-AUC": tuple(fold_pr_auc),
            "Mean CV PR-AUC": mean_pr_auc,
            "Std CV PR-AUC": std_pr_auc,
            "Mean CV ROC-AUC": mean_roc_auc,
            "MLflow Run ID": cv_run_ids[feature_set],
        }
    )
cv_results = (
    pd.DataFrame(cv_rows).sort_values("Mean CV PR-AUC", ascending=False).reset_index(drop=True)
)
winner = cv_results.iloc[0]
display(cv_results)
print(f"TRAIN-CV winner: {winner['Feature Set']}")

2026/10/07 08:45:03 INFO mlflow.tracking.fluent: Experiment with name 'fraud-detection-composite-features' does not exist. Creating a new experiment.


fold=1 feature_set=baseline PR-AUC=0.544091


fold=1 feature_set=baseline_plus_composite_frequency PR-AUC=0.547756


fold=1 feature_set=baseline_plus_composite_history PR-AUC=0.548773


fold=1 feature_set=baseline_plus_all_composite PR-AUC=0.554229


...

fold=3 feature_set=baseline_plus_composite_frequency PR-AUC=0.625902


fold=3 feature_set=baseline_plus_composite_history PR-AUC=0.618456


fold=3 feature_set=baseline_plus_all_composite PR-AUC=0.625544


,Feature Set,Feature Count,Fold PR-AUC,Mean CV PR-AUC,Std CV PR-AUC,Mean CV ROC-AUC,MLflow Run ID
0,baseline_plus_all_composite,463,"(0.5542290843797867, 0.5748513290293689, 0.625...",0.584875,0.029965,0.906202,04db3f18cef642f0bc7e524a119e4e22
1,baseline_plus_composite_frequency,447,"(0.5477563961787254, 0.5699232299624517, 0.625...",0.581194,0.032883,0.908516,d0e6018a4c8b4dc5902dcdbe1dc5b40a
2,baseline_plus_composite_history,459,"(0.5487729527763907, 0.5524030656314062, 0.618...",0.573211,0.032028,0.901608,0d55432e7a2c459a83204032def3c87c
3,baseline,443,"(0.5440914629804267, 0.5423442441219867, 0.621...",0.569162,0.036698,0.906386,6784c22734834c22b7a1803e83dfdc74


TRAIN-CV winner: baseline_plus_all_composite


## One external VALIDATION evaluation and conclusion

Only the TRAIN-CV winner is now rebuilt on full TRAIN and evaluated once on external VALIDATION. The 0.50 threshold remains descriptive and is not optimized.

In [5]:
validation = split.validation
validation_started = time.perf_counter()
(
    x_train,
    x_validation,
    final_indices_by_set,
    final_names_by_set,
    final_feature_seconds,
    final_memory_mib,
) = prepare_feature_pair(train, validation)
winning_name = str(winner["Feature Set"])
selected = final_indices_by_set[winning_name]
final_model = lgb.LGBMClassifier(**lightgbm_parameters)
final_model.fit(x_train[:, selected], train["isFraud"].astype("int8"))
validation_probabilities = final_model.predict_proba(x_validation[:, selected])[:, 1]
validation_metrics = evaluate_probabilities(
    validation["isFraud"].astype("int8"), validation_probabilities
)
validation_seconds = time.perf_counter() - validation_started
with mlflow.start_run(run_name="composite-final-winner") as run:
    mlflow.set_tags(
        {
            "stage": "composite-feature-engineering",
            "dataset": "ieee-cis",
            "selection": "temporal-cv",
            "final_refit": "true",
        }
    )
    mlflow.log_params(
        {
            "composite_feature_set": winning_name,
            "composite_identities": json.dumps(identity_definitions),
            "feature_count": len(final_names_by_set[winning_name]),
            **lightgbm_parameters,
        }
    )
    mlflow.log_metrics(
        {
            "validation_pr_auc": validation_metrics.pr_auc,
            "validation_roc_auc": validation_metrics.roc_auc,
            "validation_precision_at_0_50": validation_metrics.precision,
            "validation_recall_at_0_50": validation_metrics.recall,
            "validation_f1_at_0_50": validation_metrics.f1,
        }
    )
    final_run_id = run.info.run_id

previous_pr_auc = 0.593938
absolute_change = validation_metrics.pr_auc - previous_pr_auc
relative_change = absolute_change / previous_pr_auc
total_seconds = time.perf_counter() - notebook_started
frequency_features = CompositeFeatureEngineer().feature_groups_["composite_frequency"]
history_features = CompositeFeatureEngineer().feature_groups_["composite_history"]


def markdown_cv_table(frame):
    rows = [
        "| Feature set | Features | Fold PR-AUCs | Mean | Std | Mean ROC-AUC |",
        "| --- | ---: | --- | ---: | ---: | ---: |",
    ]
    for _, row in frame.iterrows():
        folds_text = " / ".join(f"{value:.6f}" for value in row["Fold PR-AUC"])
        rows.append(
            f"| {row['Feature Set']} | {int(row['Feature Count'])} | {folds_text} | "
            f"{row['Mean CV PR-AUC']:.6f} | {row['Std CV PR-AUC']:.6f} | "
            f"{row['Mean CV ROC-AUC']:.6f} |"
        )
    return "\n".join(rows)


identity_lines = "\n".join(
    f"- **{name}:** {' + '.join(columns)}" for name, columns in identity_definitions.items()
)
feature_lines = "\n".join(f"- `{feature}`" for feature in (*frequency_features, *history_features))
run_lines = "\n".join(f"- **{name}:** `{run_id}`" for name, run_id in cv_run_ids.items())
change_word = "improvement" if absolute_change > 0 else "degradation"
report = f"""# Composite Entity / Identity Features

## Composite identities

{identity_lines}

Each component uses `M` when missing. Nonmissing values use `V<length>:<value>` before
joining, so a real value cannot collide with the missing token or component delimiters.

## New features

{feature_lines}

There are **{len(frequency_features) + len(history_features)}** controlled composite features:
{len(frequency_features)} fold-local frequencies and
{len(history_features)} past-only history features.
Historical batches use only `TransactionDT < T`; equal-time transactions never see each other.
No target statistic is used.

## Temporal CV

Three expanding folds inside TRAIN use fold-local frequency maps, historical state, imputation,
and preprocessing. External VALIDATION is not used for feature selection.

{markdown_cv_table(cv_results)}

- TRAIN-CV winner: **{winning_name}**
- Winner feature count: **{len(final_names_by_set[winning_name])}**

## External validation

- PR-AUC: **{validation_metrics.pr_auc:.6f}**
- ROC-AUC: **{validation_metrics.roc_auc:.6f}**
- Precision at 0.50: **{validation_metrics.precision:.6f}**
- Recall at 0.50: **{validation_metrics.recall:.6f}**
- F1 at 0.50: **{validation_metrics.f1:.6f}**
- Confusion matrix: **{validation_metrics.confusion_matrix}**
- Previous preferred LightGBM PR-AUC: **{previous_pr_auc:.6f}**
- Absolute {change_word}: **{absolute_change:+.6f}**
- Relative {change_word}: **{relative_change:+.2%}**

## Runtime, memory, and serving complexity

- Fold feature-preparation seconds: **{", ".join(f"{value:.1f}" for value in preparation_seconds)}**
- Final TRAIN/VALIDATION feature preparation: **{final_feature_seconds:.1f} seconds**
- Peak prepared fold estimate: **{max([*prepared_memory_mib, final_memory_mib]):.1f} MiB**
- Final winner fit/evaluation workflow: **{validation_seconds:.1f} seconds**
- Total notebook time before report write: **{total_seconds:.1f} seconds**
- Serving requires deterministic composite normalization, four reference maps when selected,
  and keyed count/sum/max state for selected history features. This is moderate stateful complexity.

## MLflow

- Experiment: `fraud-detection-composite-features`
{run_lines}
- **Final winner:** `{final_run_id}`

## Scope confirmation

External VALIDATION was evaluated once after TRAIN-CV selection. HOLDOUT was never assigned,
transformed, predicted, summarized, or evaluated. No graph features, CatBoost, target encoding,
calibration, threshold optimization, SHAP, or rules are included.
"""
report_path = PROJECT_ROOT / "reports" / "composite_identity_features_summary.md"
report_path.write_text(report, encoding="utf-8")
print(f"Winner: {winning_name}")
print(f"Validation PR-AUC: {validation_metrics.pr_auc:.6f}")
print(f"Absolute change: {absolute_change:+.6f}")
print(f"Relative change: {relative_change:+.2%}")
print(f"Final MLflow run: {final_run_id}")
print(f"Report: {report_path.relative_to(PROJECT_ROOT)}")
print("External validation used for selection: False")
print("Final holdout status: not assigned or accessed")

Winner: baseline_plus_all_composite
Validation PR-AUC: 0.595387
Absolute change: +0.001449
Relative change: +0.24%
Final MLflow run: a7cc25894a484b078d476e063eeb0007
Report: reports\composite_identity_features_summary.md
External validation used for selection: False
Final holdout status: not assigned or accessed
